# **Дисперсионный анализ (ANOVA): Оценка влияния факторов на клинические показатели**

* __Цель дисперсионного анализа:__ Определить, существуют ли статистически значимые различия в средних значениях ключевых физиологических показателей в зависимости от одного или нескольких категориальных факторов, а также оценить их совместное (перекрестное) влияние.
* __Задачи дисперсионного анализа:__ Оценить изолированное влияние независимых переменных на зависимую непрерывную переменную. Выявить наличие эффекта взаимодействия факторов. Доказать математически, что наблюдаемые различия между группами носят системный характер, а не являются следствием случайности выборки.
* __Алгоритм использования:__
  1. __Диагностика данных:__ Проверка обязательных статистических допущений для применения дисперсионного анализа (нормальность распределения остатков и гомоскедастичность — однородность дисперсий).
  2. __Однофакторный анализ (One-Way ANOVA):__ Выдвижение гипотез ($H_0$ и $H_1$) и оценка влияния единичного фактора (кластера) на показатель. При выявлении статистически значимых различий — проведение апостериорного анализа (Post-Hoc, критерий Тьюки) для детального попарного сравнения групп.
  3. __Двухфакторный анализ (Two-Way ANOVA):__ Оценка одновременного влияния двух факторов (например, кластера и пола). Построение конкурирующих линейных моделей (аддитивной и мультипликативной с эффектом взаимодействия) с последующим выбором оптимальной модели на основе информационного критерия Акаике (AIC).

In [ ]:
# --- ИМПОРТ БИБЛИОТЕК И ПЕРВИЧНАЯ НАСТРОЙКА ---

# Сторонние библиотеки
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Локульные модули
from scripts.logger import setup_logger

# Инициализация логгера
logger = setup_logger("anova")

# Установка уровня логирования для предупреждений от matplotlib
logging.getLogger("matplotlib.category").setLevel(logging.WARNING)

# Настройка визуализации
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Константы
ALPHA_LEVEL = 0.05  # Уровень статистической значимости для ANOVA

# Цветовая схема для кластеров
CLUSTER_COLORS = {
    "1": "#A9DFBF",  # Зеленая зона (Низкий риск)
    "2": "#F9E79F",  # Желтая зона (Умеренный риск / Пограничное состояние)
    "3": "#F5B7B1",  # Красная зона (Высокий риск / Патология)
}

In [ ]:
# --- ЗАГРУЗКА ДАННЫХ И ПЕРВИЧНЫЙ АНАЛИЗ ---

try:
    data = pd.read_csv("../data/processed/heart_disease_uci_clustered.csv")
    data["Cluster_Hierarchical"] = (
        data["Cluster_Hierarchical"].astype(str).astype("category")
    )

    display(Markdown("#### **Содержание набора данных с метками кластеров**"))
    display(data.head())
    display(
        Markdown(
            f"Размерность данных: **{data.shape[0]} строк и {data.shape[1]} столбцов.**"
        )
    )
except FileNotFoundError as e:
    logger.error(
        "Файл `../data/processed/heart_disease_uci_clustered.csv` не найден. Убедитесь, что пайплайн кластеризации был выполнен."
    )
    raise e

## **1. Однофакторный дисперсионный анализ (One-way ANOVA)**

### **1.1. Визуализация распределений**

* __Цель:__ Наглядно оценить распределение непрерывной переменной внутри каждой категории (кластера) перед проведением строгих математических тестов.
* __Задачи:__ Визуально сравнить медианы, оценить внутригрупповой разброс (дисперсию), выявить наличие возможных выбросов и проанализировать пересечение доверительных интервалов средних значений.
* __Алгоритм использования:__
  1. __Построение ящика с усами (Boxplot):__ Используется для оценки формы распределения, межквартильного размаха и фиксации аномалий.
  2. __Построение точечной диаграммы (Pointplot / Mean plot):__ Используется для фокусировки на средних арифметических значениях и визуализации их 95% доверительных интервалов (95% CI). Если доверительные интервалы не пересекаются, это визуальный маркер статистически значимых различий.

In [ ]:
# --- 1.1. ВИЗУАЛИЗАЦИЯ РАСПРЕДЕЛЕНИЙ ---

display(
    Markdown(
        "### **Визуализация распределений: Ящик с усами (Boxplot) и Точечная диаграмма (Pointplot)**"
    )
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# График 1: Boxplot
sns.boxplot(
    x="Cluster_Hierarchical",
    y="thalch",
    hue="Cluster_Hierarchical",
    data=data,
    order=["1", "2", "3"],
    palette=CLUSTER_COLORS,
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Boxplot: Распределение thalch", fontsize=14)
axes[0].set_xlabel("Клинический профиль (Кластер)", fontsize=12)
axes[0].set_ylabel("Средняя Макс. ЧСС (thalch)", fontsize=12)

# График 2: Pointplot
sns.pointplot(
    x="Cluster_Hierarchical",
    y="thalch",
    data=data,
    order=["1", "2", "3"],
    capsize=0.1,
    markers="o",
    linestyles="-",
    legend=False,
    ax=axes[1],
)

if axes[1].lines:
    for line in axes[1].lines:
        line.set_color("#333333")

axes[1].set_title("Pointplot: Средние значения с 95% дов. интервалом", fontsize=14)
axes[1].set_xlabel("Клинический профиль (Кластер)", fontsize=12)
axes[1].set_ylabel("Средняя Макс. ЧСС (thalch)", fontsize=12)

fig.suptitle("Визуальная оценка распределения thalch по кластерам", fontsize=16)
plt.tight_layout()
plt.show()

### **1.2. Проверка статистических допущений (Assumptions Check)**

* __Цель:__ Убедиться в математической обоснованности и корректности применения классического параметрического дисперсионного анализа (F-критерия) к текущему набору данных.
* __Задачи:__ Подтвердить, что данные в каждой группе распределены нормально, а их дисперсии (разброс) статистически равны.
* __Алгоритм использования:__
  1. __Проверка гомоскедастичности (однородности дисперсий):__ Применение критерия Левена (Levene's test), который менее чувствителен к отклонениям от идеального нормального распределения. Нулевая гипотеза предполагает равенство дисперсий.
  2. __Проверка нормальности распределения остатков:__ Визуальная диагностика с помощью графика квантилей (Q-Q Plot), где точки должны выстраиваться вдоль диагональной опорной линии , и формальное математическое подтверждение с помощью критерия Шапиро-Уилка (Shapiro-Wilk test).

In [ ]:
# --- 1.2. ПРОВЕРКА СТАТИСТИЧЕСКИХ ДОПУЩЕНИЙ (ASSUMPTIONS CHECK) ---

display(Markdown("### **Проверка статистических допущений (Assumptions Check)**"))

# Подготовка массивов данных по группам (исключая NaN)
clean_data = data.dropna(subset=["thalch", "Cluster_Hierarchical"]).copy()
groups = [
    clean_data[clean_data["Cluster_Hierarchical"] == str(i)]["thalch"]
    for i in range(1, 4)
]

# 1. Проверка гомоскедастичности (Тест Левена)
stat_levene, p_levene = stats.levene(*groups)
levene_result = (
    "Дисперсии однородны (`H0` не отвергается)."
    if p_levene > ALPHA_LEVEL
    else "Дисперсии неоднородны (`H0` отвергается)."
)
display(
    Markdown(
        f"#### **1. Проверка гомоскедастичности (Тест Левена)**\n* **Statistic:** {stat_levene:.3f}, **p-value:** `{p_levene:.4e}`\n\n* **Вывод:** {levene_result}"
    )
)

# 2. Проверка нормальности остатков (OLS и Шапиро-Уилк)
model = ols("thalch ~ C(Cluster_Hierarchical)", data=clean_data).fit()
residuals = model.resid

stat_shapiro, p_shapiro = stats.shapiro(residuals)
shapiro_result = (
    "Остатки распределены нормально."
    if p_shapiro > ALPHA_LEVEL
    else "Распределение остатков отличается от нормального."
)
display(
    Markdown(
        f"#### **2. Проверка нормальности остатков (Тест Шапиро-Уилка)**\n* **W:** {stat_shapiro:.3f}, **p-value:** `{p_shapiro:.4e}`\n\n* **Вывод:** {shapiro_result}"
    )
)

# Q-Q Plot
display(Markdown("#### **Q-Q Plot для визуальной оценки нормальности остатков**"))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
sm.qqplot(
    residuals,
    fit=True,
    line="45",
    ax=ax,
    marker="o",
    markerfacecolor="none",
    markeredgecolor="#333333",
    alpha=0.8,
)

if len(ax.lines) > 1:
    ax.lines[1].set_color("black")
    ax.lines[1].set_linestyle("--")
    ax.lines[1].set_linewidth(1.5)

ax.set_title("Normal Q-Q Plot", fontsize=14)
plt.tight_layout()
plt.show()

### **1.3. Построение модели ANOVA (One-Way ANOVA)**

* __Цель:__ Количественно доказать или опровергнуть наличие статистически значимых различий в средних значениях зависимой переменной между исследуемыми группами.
* __Задачи:__ Рассчитать значение F-критерия и соответствующее ему p-value. В случае нарушения базовых статистических допущений (например, нормальности) — подтвердить надежность результатов с помощью альтернативного математического аппарата.
* __Алгоритм использования:__
  1. __Расчет классического дисперсионного анализа:__ Построение модели и оценка F-статистики. Если p-value меньше заданного уровня значимости ($\alpha$), нулевая гипотеза отвергается.
  2. __Проверка на устойчивость (Robustness Check):__ Проведение непараметрического теста (например, критерия Краскела-Уоллиса), который оперирует не средними значениями, а рангами, и не требует идеального нормального распределения данных.

In [ ]:
# --- 1.3. ПОСТРОЕНИЕ МОДЕЛИ ANOVA ---

# 1. Проверка значимости модели ANOVA
display(
    Markdown(
        "### **Построение модели ANOVA (One-Way ANOVA)**\n#### **Таблица дисперсионного анализа (ANOVA)**"
    )
)

anova_table = sm.stats.anova_lm(model, typ=2)
display(anova_table)

p_anova = anova_table.loc["C(Cluster_Hierarchical)", "PR(>F)"]
if p_anova < ALPHA_LEVEL:
    display(
        Markdown(
            f"* **p-value ANOVA:** `{p_anova:.4e}`\n\n* **Вывод:** Отвергаем нулевую гипотезу (`H0`). Существуют статистически значимые различия в средней макс. ЧСС (`thalch`) между кластерами (при $\\alpha={ALPHA_LEVEL}$)."
        )
    )
else:
    display(
        Markdown(
            f"* **p-value ANOVA:** `{p_anova:.4e}`\n\n* **Вывод:** Нет статистически значимых различий между группами (при $\\alpha={ALPHA_LEVEL}$)."
        )
    )

# 2. Проверка на устойчивость (Robustness Check)
stat_kw, p_kw = stats.kruskal(*groups)
kw_result = "полностью подтверждает" if p_kw < ALPHA_LEVEL else "не подтверждает"
display(
    Markdown(
        f"#### **Проверка на устойчивость: Непараметрический тест Краскела-Уоллиса**\n* **H-statistic:** {stat_kw:.3f}, **p-value:** `{p_kw:.4e}`\n\n* **Вывод:** Непараметрический тест {kw_result} результаты ANOVA."
    )
)

### **1.4. Апостериорный анализ (Post-Hoc)**

* __Цель:__ Точно определить, между какими именно парами категорий (кластеров) существуют статистически значимые различия.
* __Задачи:__ Провести множественные попарные сравнения групп, строго контролируя групповую вероятность ошибки первого рода (Family-Wise Error Rate, FWER) — то есть вероятность найти ложные различия там, где их нет.
* __Алгоритм использования:__
  1. __Логический барьер:__ Проверка результатов пункта 1.3. Апостериорный анализ проводится только в том случае, если глобальный тест ANOVA показал наличие значимых различий.
  2. __Расчет критерия Тьюки (Tukey HSD):__ Попарное сравнение всех групп с автоматической корректировкой p-value.
  3. __Визуализация доверительных интервалов:__ Построение графика Тьюки. Если доверительный интервал разницы средних для конкретной пары не пересекает нулевую отметку, различия признаются статистически значимыми.

In [ ]:
# --- 1.4. АПОСТЕРИОРНЫЙ АНАЛИЗ (POST-HOC) ---

display(Markdown("### **Апостериорный анализ (Post-Hoc)**"))

if p_anova < ALPHA_LEVEL:
    # Расчет критерия Тьюки
    display(
        Markdown(
            "#### **Множественные попарные сравнения: Критерий Тьюки (Tukey HSD)**"
        )
    )
    tukey = pairwise_tukeyhsd(
        endog=clean_data["thalch"],
        groups=clean_data["Cluster_Hierarchical"],
        alpha=ALPHA_LEVEL,
    )
    display(tukey.summary())

    # Визуализация доверительных интервалов Тьюки
    display(Markdown("#### **Визуализация: График доверительных интервалов Тьюки**"))
    fig = tukey.plot_simultaneous(
        figsize=(10, 6),
        xlabel="Максимальная ЧСС (thalch)",
        ylabel="Клинический профиль (Кластер)",
    )
    fig.axes[0].set_title(
        "Доверительные интервалы для разницы средних (Tukey HSD)", fontsize=14
    )
    plt.tight_layout()
    plt.show()
else:
    display(
        Markdown(
            "*Апостериорный анализ не проводится, так как глобальный тест ANOVA не выявил статистически значимых различий между группами.*"
        )
    )

### **1.5. Анализ и интерпретация результатов однофакторного дисперсионного анализа (One-way ANOVA)**

**Аналитическое заключение по результатам тестирования:**
На данном этапе исследования была проведена строгая статистическая оценка влияния принадлежности пациента к определенному клиническому профилю (Кластеру) на его способность достигать максимальной частоты сердечных сокращений (`thalch`) в момент физической нагрузки.

**Ключевые результаты оценки:**

1. **Глобальные различия подтверждены (ANOVA):** Классический дисперсионный анализ показал экстремально низкое значение p-value ($1.16 \times 10^{-57}$), что позволяет с абсолютной уверенностью отвергнуть нулевую гипотезу. Различия в максимальном пульсе между кластерами не случайны — они носят ярко выраженный системный характер.
2. **Проверка на устойчивость пройдена:** Непараметрический тест Краскела-Уоллиса ($p < 0.0001$) полностью подтвердил выводы классического F-критерия, доказав, что незначительные отклонения распределения остатков от идеального нормального (зафиксированные тестом Шапиро-Уилка) не исказили итоговый математический результат.
3. **Абсолютная уникальность каждого кластера (Критерий Тьюки):** Апостериорный анализ доказал, что статистически значимая разница существует между всеми без исключения парами кластеров (1-2, 1-3 и 2-3). Столбец `reject = True` для каждой пары подтверждает, что алгоритм машинного обучения не создал "искусственных" или пересекающихся по этому признаку границ.

**Клинико-физиологическая интерпретация паттернов:**

* **Кластер 1 (Зеленая зона / Низкий риск):** Демонстрирует наиболее высокие средние значения максимального пульса (около 154 уд/мин). Физиологически это означает сохраненную хронотропную функцию сердца — способность миокарда адекватно разгонять ритм в ответ на стресс или нагрузку.
* **Кластеры 2 и 3 (Желтая и Красная зоны):** Демонстрируют резкое падение показателей (до 124-130 уд/мин). В кардиологии неспособность достичь 85% от возрастной нормы пульса при нагрузке называется "хронотропной недостаточностью". Это классический маркер ишемической болезни сердца, дисфункции синусового узла или приема сильных бета-блокаторов (что косвенно подтверждает тяжесть состояния этих пациентов).

**Итоговое методологическое резюме:**
Алгоритм иерархической кластеризации, обучавшийся исключительно математическим путем "без учителя", смог выделить когорты пациентов, которые имеют фундаментально разный уровень физиологического резерва сердечно-сосудистой системы. Это доказывает высокую репрезентативность и практическую ценность разработанной модели сегментации.

## **2. Двухфакторный дисперсионный анализ (Two-Way ANOVA)**

### **2.1. Визуальная оценка эффекта взаимодействия**

* __Цель:__ Предварительно (до строгих математических расчетов) оценить характер совместного влияния двух независимых категориальных факторов на непрерывную зависимую переменную и выявить признаки их перекрестного влияния.
* __Задачи:__ Сравнить медианы и дисперсии внутри образованных подгрупп. Визуально определить, зависит ли влияние одного фактора от уровня другого фактора.
* __Алгоритм использования:__
  1. __Построение сгруппированного ящика с усами (Grouped Boxplot):__ Оценка формы распределения и наличия выбросов для каждой уникальной комбинации факторов.
  2. __Построение графика взаимодействия (Interaction Plot):__ Визуализация средних значений. Если линии тренда (например, для разных полов) идут параллельно — факторы действуют независимо друг от друга. Если линии пересекаются или имеют принципиально разный угол наклона — присутствует эффект взаимодействия.

In [ ]:
# --- 2.1. ВИЗУАЛИЗАЦИЯ: ОЦЕНКА ЭФФЕКТА ВЗАИМОДЕЙСТВИЯ ---

display(
    Markdown(
        "#### **Визуализация взаимодействия: Сгруппированный ящик с усами (Grouped Boxplot) и График взаимодействия (Interaction Plot)**"
    )
)

# Подготовка данных
clean_data_2way = data.dropna(subset=["trestbps", "Cluster_Hierarchical", "sex"]).copy()
clean_data_2way["sex"] = clean_data_2way["sex"].astype(str).astype("category")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# График 1: Сгруппированный Boxplot
sns.boxplot(
    x="Cluster_Hierarchical",
    y="trestbps",
    hue="sex",
    data=clean_data_2way,
    order=["1", "2", "3"],
    palette="Set2",
    ax=axes[0],
)
axes[0].set_title(
    "Grouped Boxplot: Артериальное давление (trestbps) по кластерам и полу (sex)",
    fontsize=14,
)
axes[0].set_xlabel("Клинический профиль (Кластер)", fontsize=12)
axes[0].set_ylabel("Артериальное давление в покое (trestbps)", fontsize=12)
axes[0].legend(title="Пол (sex)")

# График 2: График взаимодействия (Interaction Plot)
sns.pointplot(
    x="Cluster_Hierarchical",
    y="trestbps",
    hue="sex",
    data=clean_data_2way,
    order=["1", "2", "3"],
    palette="Set2",
    dodge=0.1,
    markers=["o", "s"],
    capsize=0.1,
    linestyles=["-", "--"],
    ax=axes[1],
)
axes[1].set_title("Interaction Plot: Оценка перекрестного влияния", fontsize=14)
axes[1].set_xlabel("Клинический профиль (Кластер)", fontsize=12)
axes[1].set_ylabel("Среднее артериальное давление (trestbps)", fontsize=12)
axes[1].legend(title="Пол (sex)")

fig.suptitle(
    "Визуальная оценка влияния кластера и пола (sex) на артериальное давление (trestbps)",
    fontsize=16,
)
plt.tight_layout()
plt.show()

### **2.2. Построение моделей и выбор по критерию Акаике (AIC)**

* __Цель:__ Определить оптимальную математическую спецификацию модели дисперсионного анализа, которая наилучшим образом описывает данные, соблюдая баланс между точностью предсказаний и простотой.
* __Задачи:__ Сравнить модель, предполагающую независимое влияние факторов, с моделью, учитывающей их синергетический (перекрестный) эффект. Вычислить итоговые p-values для статистически обоснованных выводов.
* __Алгоритм использования:__
  1. __Построение аддитивной модели ($Y \sim A + B$):__ Факторы добавляются в уравнение изолированно.
  2. __Построение модели с взаимодействием ($Y \sim A * B$):__ В уравнение добавляется дополнительный член ($A \times B$), учитывающий перекрестное влияние факторов.
  3. __Сравнение по информационному критерию Акаике (AIC):__ Модель с наименьшим значением AIC признается победителем.
  4. __Расчет F-статистики:__ Построение итоговой таблицы ANOVA на базе победившей модели для оценки статистической значимости каждого фактора (p-value $< \alpha$).

In [ ]:
# --- 2.2. ПОСТРОЕНИЕ МОДЕЛЕЙ, ВЫБОР ПО AIC И РАСЧЕТ ANOVA ---

# 1. Построение конкурирующих моделей
# Модель 1: Аддитивная (без взаимодействия) -> Y ~ A + B
model_add = ols(
    "trestbps ~ C(Cluster_Hierarchical) + C(sex)", data=clean_data_2way
).fit()

# Модель 2: С взаимодействием -> Y ~ A * B (эквивалентно Y ~ A + B + A:B)
model_int = ols(
    "trestbps ~ C(Cluster_Hierarchical) * C(sex)", data=clean_data_2way
).fit()

# Сравнение по критерию Акаике (AIC)
aic_add, aic_int = model_add.aic, model_int.aic
is_interaction_better = aic_int < aic_add
best_model = model_int if is_interaction_better else model_add

aic_conclusion = (
    "* **Вывод:** Модель **с взаимодействием** (Interaction Model) описывает данные лучше (AIC ниже)."
    if is_interaction_better
    else "* **Вывод:** **Аддитивная** модель (Additive Model) описывает данные лучше. Эффект взаимодействия незначим."
)

display(
    Markdown(
        f"#### **Сравнение конкурирующих моделей по критерию Акаике (AIC)**\n"
        f"* **AIC Аддитивной модели:** `{aic_add:.2f}`\n\n"
        f"* **AIC Модели с взаимодействием:** `{aic_int:.2f}`\n\n"
        f"{aic_conclusion}"
    )
)

# 2. Итоговая таблица Two-Way ANOVA
display(Markdown("#### **Таблица двухфакторного дисперсионного анализа (ANOVA)**"))
anova_2way = sm.stats.anova_lm(best_model, typ=2)
display(anova_2way)

# 3. Клинико-статистическая интерпретация
p_cluster = anova_2way.get("PR(>F)", {}).get("C(Cluster_Hierarchical)", np.nan)
p_sex = anova_2way.get("PR(>F)", {}).get("C(sex)", np.nan)

results_text = "#### **Клинико-статистическая интерпретация**\n"
if pd.notna(p_cluster):
    results_text += f"* **Влияние профиля (Кластера):** p-value = `{p_cluster:.4e}` $\\rightarrow$ **{'ЗНАЧИМО' if p_cluster < ALPHA_LEVEL else 'НЕ ЗНАЧИМО'}**\n\n"
if pd.notna(p_sex):
    results_text += f"* **Влияние пола (sex):** p-value = `{p_sex:.4e}` $\\rightarrow$ **{'ЗНАЧИМО' if p_sex < ALPHA_LEVEL else 'НЕ ЗНАЧИМО'}**\n\n"

# Проверка наличия эффекта взаимодействия
interaction_idx = "C(Cluster_Hierarchical):C(sex)"
if interaction_idx in anova_2way.index:
    p_int = anova_2way.loc[interaction_idx, "PR(>F)"]
    results_text += f"* **Эффект взаимодействия:** p-value = `{p_int:.4e}` $\\rightarrow$ **{'ЗНАЧИМ' if p_int < ALPHA_LEVEL else 'НЕ ЗНАЧИМ'}**\n"

display(Markdown(results_text))

### **2.3. Анализ и интерпретация результатов двухфакторного дисперсионного анализа (Two-Way ANOVA)**

**Аналитическое заключение по результатам моделирования:**
На данном этапе оценивалось одновременное влияние клинического профиля (Кластера) и пола пациента (`sex`) на уровень базового артериального давления в покое (`trestbps`), а также проверялось наличие синергетического (перекрестного) эффекта между этими факторами.

**Ключевые результаты оценки:**

1. **Отсутствие эффекта взаимодействия (AIC):** Информационный критерий Акаике показал, что аддитивная модель (AIC = 7611.53) описывает данные лучше, чем модель с взаимодействием (AIC = 7614.96). Это математически доказывает, что влияние клинического кластера на артериальное давление не зависит от пола пациента. График `Interaction Plot` визуально подтверждает этот вывод: линии тренда для мужчин и женщин практически параллельны.
2. **Доминирующее влияние кластера:** Принадлежность к клиническому профилю оказывает статистически высокозначимое влияние на давление (p-value = `3.32e-07`). Это подтверждает, что кластеры надежно разделяют пациентов по степени гипертензии.
3. **Отсутствие независимого влияния пола:** Фактор пола не преодолел порог статистической значимости (p-value = `0.468`). Различия в артериальном давлении между мужчинами и женщинами в рамках нашей выборки (с поправкой на кластер) носят исключительно случайный характер.

**Клинико-физиологическая интерпретация паттернов:**

Вне зависимости от гендерной принадлежности, пациенты демонстрируют идентичную траекторию изменения артериального давления. Наивысшие показатели фиксируются в "Красной зоне" (Кластер 3), что логично укладывается в общую картину высокого риска. Динамика сосудистых реакций при переходе между кластерами является универсальным физиологическим маркером, не имеющим выраженной специфики по полу в исследуемой популяции.

**Итоговое методологическое резюме:**
Двухфакторный дисперсионный анализ в очередной раз подтвердил робастность (надежность) нашей иерархической кластеризации. Выделенные алгоритмом сегменты остаются мощнейшим и независимым предиктором состояния сердечно-сосудистой системы даже при включении в модель побочных демографических ковариат.